# 1) Data Contract (Lane 2: Refresh / Content Opportunity Scoring)

1. **Unit of Analysis / Grain:** One row = One unique content item (`content_hash_id`).
2. **Tables Used:** `dim_content` joined with `fact_content_daily_performance` (or `fact_content_daily_performance_sample`).
3. **Time Window:** Mid-panel observation window (e.g., March 2026: `2026-03-01` to `2026-03-31`).
4. **Target / Proxy:** `is_declining_label` (defined as directional traffic drop over a 30-day forward window).
5. **Deliberately Excluded Field:** `health_score` / product decision flags (excluded to prevent circular logic and policy replication leakage).

In [ ]:
import os
import pandas as pd

# Load token from environment file
hf_token = os.getenv("HF_TOKEN", "")

# Load starter dataset for verification queries
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("--- Query 1: Grain Check ---")
grain_check = len(df) == df['content_id'].nunique()
print(f"Is content_id unique (One row = One page)? {grain_check} (Total rows: {len(df):,})")

print("\n--- Query 2: Slice Row Count & Target Volume ---")
total_rows = len(df)
declining_rows = len(df[df['trend_direction'] == 'down'])
print(f"Total Rows: {total_rows:,} | Declining Rows ('down'): {declining_rows:,}")

print("\n--- Query 3: Availability Check (IS TRUE) ---")
surviving_rows = len(df[df['impressions_90d'] > 0])
print(f"Rows surviving minimum volume filter (impressions > 0): {surviving_rows:,}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Build 5-Feature Frame (All knowable at prediction time)
features = ['impressions_90d', 'clicks_90d', 'content_age_days', 'avg_position', 'ctr']
X = df[features].fillna(0)
y = (df['trend_direction'] == 'down').astype(int)

# 2. Honest Model Baseline
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X, y)
preds = model.predict(X)
honest_precision = precision_score(y, preds)
print(f"Honest Feature Set Precision: {honest_precision:.4f}")

# 3. THE TRAP: Add a label-derived leaking feature on purpose
df['LEAK_trend_down_flag'] = (df['trend_direction'] == 'down').astype(int)

X_leaked = df[features + ['LEAK_trend_down_flag']].fillna(0)
model_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
model_leaked.fit(X_leaked, y)
leaked_precision = precision_score(y, model_leaked.predict(X_leaked))

print(f"LEAKED Feature Set Precision: {leaked_precision:.4f} (Artificial Jump toward 1.0!)")

# 4. Remove the leaking feature to keep the model honest
df.drop(columns=['LEAK_trend_down_flag'], inplace=True)
print("Leaking column successfully removed! Honest numbers restored.")

# 4) Named Limitation of this Slice

**Limitation:** The anonymized starter dataset represents an unbalanced snapshot. Because history varies by client (some GA4/GSC connections started at different dates), time-series features over a fixed window can exhibit varying sparsity across different client domains.